In [1]:
import sys, pathlib
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))
from db.dataraflow_db import run_query
import pandas as pd

In [ ]:
### Step 2: Exploring Individual Tables
# Query the first 5 rows of the 2013 World Bank GDP table from the week08 schema
gdp_preview = run_query("SELECT TOP 5 * FROM week08.wb_gdp_2013")
display(gdp_preview)

# Query the first 5 rows of the 2013 World Bank Population table from the week08 schema
pop_preview = run_query("SELECT TOP 5 * FROM week08.wb_pop_2013")
display(pop_preview)

,country,year,NY.GDP.MKTP.CD
0,Arab World,2013,2.827557e+12
1,Caribbean small states,2013,6.928705e+10
2,Central Europe and the Baltics,2013,1.419384e+12
3,East Asia & Pacific (all income levels),2013,2.084673e+13
4,East Asia & Pacific (developing only),2013,1.172940e+13


,country,year,SP.POP.TOTL
0,Arab World,2013,3.770967e+08
1,Caribbean small states,2013,6.975819e+06
2,Central Europe and the Baltics,2013,1.037137e+08
3,East Asia & Pacific (all income levels),2013,2.248867e+09
4,East Asia & Pacific (developing only),2013,2.006073e+09


In [ ]:
### Step 3: Performing an Inner Join in SQL
# Perform an Inner Join between GDP and Population tables using 'country' as the shared key.
# Aliases (g and p) are used to make table references clean and concise.
sql_inner_join = """
    SELECT 
        g.country, 
        g.[NY.GDP.MKTP.CD] AS gdp_usd, 
        p.[SP.POP.TOTL] AS population
    FROM week08.wb_gdp_2013 AS g
    JOIN week08.wb_pop_2013 AS p 
        ON g.country = p.country
"""

# Execute the query and load the result into a Pandas DataFrame
gdp_pop_df = run_query(sql_inner_join)

# Display the first few rows of the successfully joined dataset
gdp_pop_df.head()

,country,gdp_usd,population
0,Afghanistan,2.045894e+10,30682500.0
1,Albania,1.278103e+10,2897366.0
2,Algeria,2.097035e+11,38186135.0
3,American Samoa,NaN,55302.0
4,Andorra,3.249101e+09,75902.0


In [ ]:
### Step 4: Calculating Derived Metrics (GDP per Capita)
# Calculate GDP per capita by dividing GDP by population. 
# NULLIF is used defensively to prevent division-by-zero errors if population is 0.
sql_calculated_metric = """
    SELECT 
        g.country,
        g.[NY.GDP.MKTP.CD] AS gdp_usd,
        p.[SP.POP.TOTL] AS population,
        g.[NY.GDP.MKTP.CD] / NULLIF(p.[SP.POP.TOTL], 0) AS gdp_per_capita
    FROM week08.wb_gdp_2013 AS g
    JOIN week08.wb_pop_2013 AS p 
        ON g.country = p.country
    ORDER BY gdp_per_capita DESC
"""

# Execute the query to get countries ranked by GDP per capita
gdp_per_capita_df = run_query(sql_calculated_metric)

# Display top 10 rows
gdp_per_capita_df.head(10)

,country,gdp_usd,population,gdp_per_capita
0,Luxembourg,6.179451e+10,543360.0,113726.638979
1,Norway,5.223491e+11,5079623.0,102832.258690
2,Qatar,2.018854e+11,2101288.0,96076.996376
3,"Macao SAR, China",5.131353e+10,568056.0,90331.819132
4,Bermuda,5.573710e+09,65001.0,85748.065414
5,Switzerland,6.849192e+11,8089346.0,84669.292937
6,Australia,1.563951e+12,23125868.0,67627.773335
7,Sweden,5.787420e+11,9600379.0,60283.245223
8,Denmark,3.358775e+11,5614932.0,59818.631528
9,Singapore,3.022459e+11,5399200.0,55979.757049
